In [1]:
import torch
import numpy as np
import json
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

In [42]:
consider_same_patients = True

In [43]:
dino_embeddings = torch.load("./dino_embeddings.pth")
dino_labels = torch.load("./dino_labels.pth")
with open('dino_file_names.json') as f:
  dino_file_names = json.load(f)

supervised_embeddings = np.load("./supervised_embeddings.npy")
supervised_labels = np.load("./supervised_labels.npy")
with open('supervised_file_names.json') as f:
  supervised_file_names = json.load(f)

In [44]:
dino_embeddings = np.asarray(dino_embeddings)
dino_labels = np.asarray(dino_labels)
dino_file_names = dino_file_names['test_file_names']

supervised_embeddings = np.asarray(supervised_embeddings)
supervised_labels = np.asarray(supervised_labels)
supervised_file_names = supervised_file_names['test_file_names']
for i, name in enumerate(supervised_file_names):
  supervised_file_names[i] = (name.split("/")[-1])[2:]

In [45]:
print(supervised_file_names[1500:1510])
print(supervised_labels[1500:1510])
print("-" * 10)
print(supervised_file_names[:10])
print(supervised_labels[:10])
print("-" * 10)
print(supervised_file_names[-10:])
print(supervised_labels[-10:])

['s5_1523_s5_14_23.png', 's5_1068_s5_11_62.png', 's5_1814_s5_16_62.png', 's5_1066_s5_11_62.png', 's5_1265_s5_11_62.png', 's5_4660_s5_9_8.png', 's5_3149_s5_2_23.png', 's5_1839_s5_16_62.png', 's5_382_s5_0_46.png', 's5_1900_s5_16_62.png']
[1 1 1 1 1 1 1 1 1 1]
----------
['s0_2557_s0_7_148.png', 's0_2062_s0_5_62.png', 's0_1371_s0_2_62.png', 's0_373_s0_0_68.png', 's0_2620_s0_8_130.png', 's0_2410_s0_7_100.png', 's0_991_s0_1_85.png', 's0_2082_s0_5_62.png', 's0_412_s0_0_68.png', 's0_766_s0_1_148.png']
[0 0 0 0 0 0 0 0 0 0]
----------
['s3_852_s3_1_23.png', 's3_1270_s3_3_148.png', 's3_946_s3_20_23.png', 's3_1011_s3_24_133.png', 's3_519_s3_10_23.png', 's3_840_s3_1_148.png', 's3_1816_s3_8_148.png', 's3_2002_s3_9_23.png', 's3_1892_s3_9_23.png', 's3_1605_s3_8_133.png']
[2 2 2 2 2 2 2 2 2 2]


In [46]:
print(dino_file_names[1500:1510])
print(dino_labels[1500:1510])
print("-" * 10)
print(dino_file_names[:10])
print(dino_labels[:10])
print("-" * 10)
print(dino_file_names[-10:])
print(dino_labels[-10:])

['s3_1925_s3_9_23.png', 's3_1926_s3_9_23.png', 's3_1927_s3_9_23.png', 's3_1929_s3_9_23.png', 's3_1934_s3_9_23.png', 's3_1935_s3_9_23.png', 's3_1936_s3_9_23.png', 's3_1939_s3_9_23.png', 's3_193_s3_0_148.png', 's3_1942_s3_9_23.png']
[1 1 1 1 1 1 1 1 1 1]
----------
['s0_1000_s0_1_85.png', 's0_1006_s0_1_85.png', 's0_1007_s0_1_85.png', 's0_1009_s0_2_100.png', 's0_1013_s0_2_100.png', 's0_1015_s0_2_100.png', 's0_1019_s0_2_100.png', 's0_1023_s0_2_100.png', 's0_1027_s0_2_100.png', 's0_1028_s0_2_100.png']
[0 0 0 0 0 0 0 0 0 0]
----------
['s5_941_s5_11_62.png', 's5_944_s5_11_62.png', 's5_948_s5_11_62.png', 's5_950_s5_11_62.png', 's5_951_s5_11_62.png', 's5_955_s5_11_62.png', 's5_962_s5_11_62.png', 's5_972_s5_11_62.png', 's5_996_s5_11_62.png', 's5_99_s5_0_13.png']
[2 2 2 2 2 2 2 2 2 2]


In [47]:
print(dino_embeddings.shape)
print(supervised_embeddings.shape)

(3000, 384)
(3000, 512)


In [48]:
counter = 0

for i in range(len(supervised_file_names)):
  supervised_file = supervised_file_names[i]
  if supervised_file in dino_file_names:
    counter += 1

print(counter)

3000


In [49]:
total_number = 0
currect_number = 0

for i in tqdm(range(len(dino_file_names))):
  # dino
  current_dino_embedding = dino_embeddings[i, :]
  current_dino_label = int(dino_labels[i])
  current_dino_file_name = dino_file_names[i]
  current_dino_patient_id = int(current_dino_file_name.split(".")[0].split("_")[-1])

  dino_knn = NearestNeighbors(n_neighbors=len(dino_embeddings), metric="euclidean")
  dino_knn.fit(dino_embeddings)
  _, dino_indices = dino_knn.kneighbors([current_dino_embedding])
  dino_sorted_indices = dino_indices.flatten()
  dino_sorted_indices = dino_sorted_indices[1:]

  # supervised
  supervised_index = supervised_file_names.index(current_dino_file_name)
  current_supervised_embedding = supervised_embeddings[supervised_index, :]
  current_supervised_label = int(supervised_labels[supervised_index])
  current_supervised_file_name = supervised_file_names[supervised_index]
  current_supervised_patient_id = int(current_supervised_file_name.split(".")[0].split("_")[-1])

  supervised_knn = NearestNeighbors(n_neighbors=len(supervised_embeddings), metric="euclidean")
  supervised_knn.fit(supervised_embeddings)
  _, supervised_indices = supervised_knn.kneighbors([current_supervised_embedding])
  supervised_sorted_indices = supervised_indices.flatten()
  supervised_sorted_indices = supervised_sorted_indices[1:]

  if consider_same_patients:
    dino_nearest_label = -1
    supervised_nearest_label = -1

    for neighbour in dino_sorted_indices:
      neighbour_patient_id = int(dino_file_names[neighbour].split(".")[0].split("_")[-1])
      if current_dino_patient_id != neighbour_patient_id:
        dino_nearest_label = int(dino_labels[neighbour])
        break

    for neighbour in supervised_sorted_indices:
      neighbour_patient_id = int(supervised_file_names[neighbour].split(".")[0].split("_")[-1])
      if current_supervised_patient_id != neighbour_patient_id:
        supervised_nearest_label = int(supervised_labels[neighbour])
        break

    if dino_nearest_label == current_dino_label or supervised_nearest_label == current_supervised_label:
      currect_number += 1
  else:
    dino_nearest_neighbour = dino_sorted_indices[0]
    dino_nearest_label = int(dino_labels[dino_nearest_neighbour])

    supervised_nearest_neighbour = supervised_sorted_indices[0]
    supervised_nearest_label = int(supervised_labels[supervised_nearest_neighbour])

    if current_dino_label == dino_nearest_label or current_supervised_label == supervised_nearest_label:
      currect_number += 1

  total_number += 1

100%|██████████| 3000/3000 [00:52<00:00, 56.78it/s]


In [50]:
currect_number / total_number

0.7663333333333333